In [ ]:
import pandas as  pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.preprocessing import OneHotEncoder
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.cluster import AgglomerativeClustering
from sklearn.cluster import DBSCAN

In [ ]:
df =  pd.read_csv("Segmentation.csv")

# PreProcessing

In [ ]:
# handle Null Values 
df["Income"] = df["Income"].fillna(df["Income"].median())

 # Feature Engineering .

In [ ]:
# Age 
df["Age"] =  2026 - df["Year_Birth"] 

In [ ]:
# Customer joining Date

df["Dt_Customer"] = pd.to_datetime(df["Dt_Customer"],dayfirst= True)

reference_date = df["Dt_Customer"].max()

df["Customer_Tenure"]= (reference_date - df["Dt_Customer"]).dt.days


In [ ]:
# Total Spending
df["T_Spending"] = df["MntWines"]+df["MntFruits"]+df["MntMeatProducts"]+df["MntFishProducts"]+df["MntSweetProducts"]+df["MntGoldProds"]

In [ ]:
#  Total Children

df["T_Children"] = df["Kidhome"]+df["Teenhome"]

In [ ]:
df["Education"] = df["Education"].replace({
    "Basic":"Undergraduate" ,
    "2n Cycle":"Undergraduate",
    "Master":"Postgraduate",
    "PhD":"Postgraduate",
    "Graduation":"Graduate"})

In [ ]:
df["Living_with"] = df["Marital_Status"].replace({
    "Married":"Partner",
    "Together":"Partner",
    "Single":"Alone",
    "Divorced":"Alone",
    "Widow":"Alone",
    "Alone":"Alone",
    "Absurd":"Alone",    
    "YOLO":"Alone"         
})


# Remove Columns  

In [ ]:
cols =["ID","Year_Birth","Marital_Status","Kidhome","Teenhome","Dt_Customer"]
Spending =["MntWines","MntFruits","MntMeatProducts","MntFishProducts","MntSweetProducts","MntGoldProds"]

cols_to_del =cols + Spending
clean_df = df.drop(columns=cols_to_del)

#  Outlier

In [ ]:
print("DF with outlier :", len(clean_df))
# Age 
Q1_age = clean_df["Age"].quantile(0.25)
Q3_age= clean_df["Age"].quantile(0.75)
IQR_age = Q3_age - Q1_age 

lower_age = (Q1_age - 1.5 * IQR_age)
upper_age = (Q3_age + 1.5 * IQR_age)

# Income

Q1_income = clean_df["Income"].quantile(0.25)
Q3_income = clean_df["Income"].quantile(0.75)
IQR_income = Q3_income - Q1_income

lower_income = Q1_income - 1.5 * IQR_income
upper_income = Q3_income + 1.5 * IQR_income

clean_df = clean_df[(clean_df["Age"] >= lower_age) & (clean_df["Age"] <= upper_age) ]
clean_df = clean_df[(clean_df["Income"] >= lower_income) & (clean_df["Income"] <= upper_income)]

print("DF without outlier :", len(clean_df))

# Encoding

In [ ]:
cat_cols=["Education","Living_with"]

ohe = OneHotEncoder()
enc_cols = ohe.fit_transform(clean_df[cat_cols])

In [ ]:
enc_df = pd.DataFrame(enc_cols.toarray(),columns = ohe.get_feature_names_out(cat_cols),index=clean_df.index)

In [ ]:
encoded_df = pd.concat([clean_df.drop(columns = cat_cols),enc_df],axis=1)

# Scaling

In [ ]:
X = encoded_df

In [ ]:
scaler = StandardScaler()
df_scale = scaler.fit_transform(X)

# PCA

In [ ]:
pca = PCA(n_components=3)
X_pca = pca.fit_transform(df_scale)

# Elbow Method

In [ ]:
wcss =[]

for k in  range(1,11):
    model = KMeans(n_clusters=k, random_state=42)
    model.fit_predict(X_pca)
    wcss.append(model.inertia_)

In [ ]:
plt.plot(range(1,11), wcss ,marker ='o')
plt.xlabel("Number of K")
plt.ylabel("Number of WCSS ")

In [ ]:
%pip install kneed

In [ ]:
from kneed import  KneeLocator

In [ ]:
knee = KneeLocator(range(1,11), wcss ,curve="convex", direction="decreasing")

In [ ]:
print(knee.elbow)

# KMeans

In [ ]:
Kmeans = KMeans(n_clusters=4,random_state=42)
K_labels = Kmeans.fit_predict(X_pca)

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    X_pca[:, 2],
    c=K_labels,
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()

# Hierarchical Clustering

In [ ]:
Aggl= AgglomerativeClustering(n_clusters = 4,linkage ='ward')
Agglo_labels = Aggl.fit_predict(X_pca)

In [ ]:
fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(111, projection="3d")

ax.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    X_pca[:, 2],
    c=Agglo_labels,
)

ax.set_xlabel("PC1")
ax.set_ylabel("PC2")
ax.set_zlabel("PC3")
plt.show()

In [ ]:
encoded_df["Clusters"] = Agglo_labels

In [ ]:
pal=["red","green","blue","Purple"]

sns.scatterplot(x=encoded_df["T_Spending"], y=encoded_df["Income"],hue=encoded_df["Clusters"],palette=pal)

In [ ]:
pal=["red","green","blue","Purple"]

sns.countplot(x=encoded_df["Clusters"],hue=encoded_df["Clusters"],palette=pal)

In [ ]:
Cluster_Summary = encoded_df.groupby("Clusters").mean()

In [ ]:
import pickle

with open("encoder.pkl", "wb") as f:
    pickle.dump(ohe, f)

with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

with open("pca.pkl", "wb") as f:
    pickle.dump(pca, f)

with open("kmeans.pkl", "wb") as f:
    pickle.dump(Kmeans, f)